# Notebook 05: Computer Vision - Object Detection

## Learning Objectives

In this notebook, you will learn:
- The difference between image classification and object detection
- How DETR (DEtection TRansformer) detects and localizes objects
- How to use the Pipeline API and manual model loading for detection
- How to visualize bounding boxes on images
- How YOLOv8 compares to DETR for real-time detection
- How to tune confidence thresholds for different use cases

## Prerequisites

### Hardware Requirements

| Model Option | Model Name | Size | Min RAM | Recommended Setup | Notes |
|--------------|------------|------|---------|-------------------|-------|
| **CPU (Small)** | facebook/detr-resnet-50 | 159MB | 8GB | 8GB RAM, CPU | Manageable on CPU |
| **GPU (Large)** | facebook/detr-resnet-101 | 232MB | 8GB | 10GB VRAM | Better accuracy |
| **YOLO (Optional)** | yolov8s.pt | 22MB | 8GB | GPU recommended | Requires `ultralytics` |

### Software Requirements
- Python 3.10+
- Libraries: `transformers`, `torch`, `PIL`
- Optional: `ultralytics` for YOLOv8 section

## Expected Behaviors

**First Time Running**:
- Model download: ~159MB for DETR-ResNet-50 (~2-3 minutes)
- Cached for future use in `~/.cache/huggingface/hub/`

**Detection Output Format**:
```python
[{'label': 'person', 'score': 0.9987, 'box': {'xmin': 50, 'ymin': 100, 'xmax': 200, 'ymax': 300}}, ...]
```

**Bounding Box Format**: `(xmin, ymin)` = top-left corner, `(xmax, ymax)` = bottom-right corner, in pixels.

**Performance Expectations**:
- CPU: 5-10 seconds per image
- GPU: 1-2 seconds per image
- Detects 80 COCO dataset categories (people, vehicles, animals, furniture, etc.)

**Confidence Thresholds**:
- threshold=0.9: Only very confident detections (may miss some objects)
- threshold=0.7: Balanced (recommended default)
- threshold=0.5: More detections but more false positives

## Overview

**Object Detection** finds and localizes multiple objects in an image, returning both class labels and bounding box coordinates.

| Task | Question Answered | Output |
|------|------------------|--------|
| Classification | What is in the image? | "cat" |
| Detection | What and where? | "cat" at (50, 100, 200, 300) |

**Use Cases**: Autonomous driving, security surveillance, retail analytics, sports analysis, medical imaging.

### DETR Architecture

DETR (DEtection TRansformer) treats object detection as a set prediction problem:
1. CNN backbone (ResNet) extracts image features
2. Transformer encoder processes spatial relationships
3. Transformer decoder predicts a fixed set of object queries
4. Bipartite matching assigns predictions to ground truth

Key advantage: no hand-designed components like anchor boxes or non-maximum suppression (NMS).

## Setup and Installation

In [ ]:
import sys, time, random, warnings
from collections import Counter
from io import BytesIO

import numpy as np
import pandas as pd
import torch
import transformers
from transformers import (
    AutoImageProcessor,
    AutoModelForObjectDetection,
    pipeline,
    set_seed,
)
from PIL import Image, ImageDraw
import requests

warnings.filterwarnings("ignore")

SEED = 1103
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
set_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"Transformers: {transformers.__version__}")
print(f"Device: {device}")

## Model Selection

In [ ]:
# Option 1: CPU-friendly (recommended for beginners)
MODEL_NAME = "facebook/detr-resnet-50"  # 159MB

# Option 2: GPU-optimized (uncomment if you have RTX 4080 or similar)
# MODEL_NAME = "facebook/detr-resnet-101"  # 232MB, better accuracy

print(f"Selected model: {MODEL_NAME}")

## Helper Functions

These utilities handle image loading and bounding box visualization across the notebook.

In [ ]:
def load_image_from_url(url: str) -> Image.Image:
    """Load an image from a URL.

    Args:
        url: HTTP(S) URL pointing to an image file.

    Returns:
        PIL Image in RGB mode.
    """
    response = requests.get(url, timeout=10)
    response.raise_for_status()
    return Image.open(BytesIO(response.content)).convert("RGB")


# Color palette for different object classes
COLORS = ["red", "blue", "green", "orange", "purple", "cyan", "magenta", "yellow"]


def draw_bounding_boxes(
    image: Image.Image, detections: list[dict], threshold: float = 0.9
) -> Image.Image:
    """Draw bounding boxes on an image.

    Args:
        image: PIL Image to annotate.
        detections: List of detection dicts with 'label', 'score', 'box' keys.
        threshold: Minimum confidence score to display.

    Returns:
        Annotated copy of the image.
    """
    img = image.copy()
    draw = ImageDraw.Draw(img)
    label_colors = {}
    color_idx = 0

    for det in detections:
        if det["score"] < threshold:
            continue

        label = det["label"]
        if label not in label_colors:
            label_colors[label] = COLORS[color_idx % len(COLORS)]
            color_idx += 1
        color = label_colors[label]

        box = det["box"]
        x1, y1, x2, y2 = box["xmin"], box["ymin"], box["xmax"], box["ymax"]
        draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
        draw.text((x1, y1 - 15), f"{label}: {det['score']:.2f}", fill=color)

    return img

## Method 1: Using Pipeline (Simplest)

The Pipeline API provides the fastest way to run object detection with just a few lines of code.

In [ ]:
try:
    print(f"Loading {MODEL_NAME}...")
    detector = pipeline(
        "object-detection",
        model=MODEL_NAME,
        device=0 if torch.cuda.is_available() else -1,
    )
    print("Model loaded successfully.")
except Exception as e:
    print(f"Error loading model: {e}")
    print("Ensure you have internet access and sufficient memory.")

### Basic Object Detection

In [ ]:
IMAGE_URL = "https://huggingface.co/datasets/huggingface/documentation-images/resolve/main/coco_sample.png"
image = load_image_from_url(IMAGE_URL)
print(f"Image size: {image.size}")

detections = detector(image, threshold=0.9)

det_df = pd.DataFrame([
    {
        "#": i,
        "Label": d["label"],
        "Score": f"{d['score']:.4f}",
        "Box (xmin, ymin, xmax, ymax)": f"({d['box']['xmin']:.0f}, {d['box']['ymin']:.0f}, {d['box']['xmax']:.0f}, {d['box']['ymax']:.0f})",
    }
    for i, d in enumerate(detections, 1)
])

print(f"\nDetected {len(detections)} objects:")
det_df

In [ ]:
# Visualize detections
image_with_boxes = draw_bounding_boxes(image, detections, threshold=0.9)
image_with_boxes

### Adjusting Confidence Threshold

The threshold controls the tradeoff between precision (fewer false positives) and recall (fewer missed objects).

In [ ]:
def compare_thresholds(
    image: Image.Image, thresholds: list[float] | None = None
) -> pd.DataFrame:
    """Compare detection counts at different confidence thresholds.

    Args:
        image: PIL Image to analyze.
        thresholds: List of threshold values to compare.

    Returns:
        DataFrame with threshold, count, and top detections.
    """
    if thresholds is None:
        thresholds = [0.5, 0.7, 0.9]

    rows = []
    for t in thresholds:
        dets = detector(image, threshold=t)
        top_labels = ", ".join(f"{d['label']} ({d['score']:.2f})" for d in dets[:5])
        rows.append({"Threshold": t, "Objects Found": len(dets), "Top Detections": top_labels})

    return pd.DataFrame(rows)


print("Threshold Comparison")
compare_thresholds(image)

## Method 2: Using Model and Processor Directly

For more control over post-processing, we can load the model and image processor separately.

In [ ]:
processor = AutoImageProcessor.from_pretrained(MODEL_NAME)
model = AutoModelForObjectDetection.from_pretrained(MODEL_NAME)
model = model.to(device)

print(f"Model loaded on: {device}")

In [ ]:
def detect_manual(
    image: Image.Image, threshold: float = 0.7
) -> pd.DataFrame:
    """Run object detection using manual model + processor.

    Args:
        image: PIL Image to analyze.
        threshold: Minimum confidence score.

    Returns:
        DataFrame with label, score, and bounding box for each detection.
    """
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.tensor([image.size[::-1]]).to(device)
    results = processor.post_process_object_detection(
        outputs, target_sizes=target_sizes, threshold=threshold
    )[0]

    rows = []
    for score, label, box in zip(results["scores"], results["labels"], results["boxes"]):
        label_name = model.config.id2label[label.item()]
        coords = box.tolist()
        rows.append({
            "Label": label_name,
            "Score": f"{score.item():.4f}",
            "Box": f"({coords[0]:.0f}, {coords[1]:.0f}, {coords[2]:.0f}, {coords[3]:.0f})",
        })

    return pd.DataFrame(rows)


test_image = load_image_from_url(
    "https://images.unsplash.com/photo-1444212477490-ca407925329e?w=600"
)

print("Manual Detection Results")
detect_manual(test_image, threshold=0.7)

## Practical Applications

These examples show common real-world detection tasks: counting objects, filtering by type, and batch processing.

### Application 1: Count Objects by Category

In [ ]:
def count_objects(
    image: Image.Image, threshold: float = 0.7
) -> pd.DataFrame:
    """Count detected objects by category.

    Args:
        image: PIL Image to analyze.
        threshold: Minimum confidence score.

    Returns:
        DataFrame with object categories and counts.
    """
    detections = detector(image, threshold=threshold)
    counts = Counter(d["label"] for d in detections)

    rows = [{"Category": label, "Count": count} for label, count in counts.most_common()]
    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Category", "Count"])


test_img = load_image_from_url(
    "https://images.unsplash.com/photo-1449965408869-eaa3f722e40d?w=600"
)

print("Object Counts (threshold=0.6)")
count_objects(test_img, threshold=0.6)

### Application 2: Filter by Object Type

In [ ]:
def find_specific_objects(
    image: Image.Image,
    target_objects: list[str],
    threshold: float = 0.7,
) -> pd.DataFrame:
    """Find specific types of objects in an image.

    Args:
        image: PIL Image to analyze.
        target_objects: List of object labels to search for.
        threshold: Minimum confidence score.

    Returns:
        DataFrame with found objects, scores, and bounding boxes.
    """
    detections = detector(image, threshold=threshold)
    filtered = [d for d in detections if d["label"] in target_objects]

    rows = []
    for d in filtered:
        box = d["box"]
        rows.append({
            "Label": d["label"],
            "Score": f"{d['score']:.3f}",
            "Box": f"({box['xmin']:.0f}, {box['ymin']:.0f}, {box['xmax']:.0f}, {box['ymax']:.0f})",
        })

    return pd.DataFrame(rows) if rows else pd.DataFrame(columns=["Label", "Score", "Box"])


test_img2 = load_image_from_url(IMAGE_URL)
print("Searching for: person, car, cup")
find_specific_objects(test_img2, target_objects=["person", "car", "cup"])

### Application 3: Multi-Image Analysis

In [ ]:
def analyze_multiple_images(
    image_urls: list[str], threshold: float = 0.8
) -> pd.DataFrame:
    """Detect objects across multiple images.

    Args:
        image_urls: List of image URLs to analyze.
        threshold: Minimum confidence score.

    Returns:
        DataFrame summarizing detections per image.
    """
    rows = []
    for i, url in enumerate(image_urls, 1):
        try:
            img = load_image_from_url(url)
            dets = detector(img, threshold=threshold)
            labels = ", ".join(d["label"] for d in dets[:5])
            rows.append({"Image": i, "Objects Found": len(dets), "Top Labels": labels})
        except Exception as e:
            rows.append({"Image": i, "Objects Found": 0, "Top Labels": f"Error: {e}"})

    return pd.DataFrame(rows)


urls = [
    "https://images.unsplash.com/photo-1606041011872-596597976b25?w=400",
    "https://images.unsplash.com/photo-1504674900247-0877df9cc836?w=400",
]

print("Multi-Image Detection")
analyze_multiple_images(urls)

## Method 3: Real-Time Detection with YOLOv8 (Optional)

YOLOv8 is the industry standard for real-time object detection. This section requires the `ultralytics` package. If not installed, you can skip this section.

In [ ]:
# YOLOv8 model family comparison
yolo_df = pd.DataFrame([
    {"Model": "YOLOv8n", "Size": "6MB", "Params": "3.2M", "mAP@50-95": 37.3, "Speed (GPU ms)": 0.99},
    {"Model": "YOLOv8s", "Size": "22MB", "Params": "11.2M", "mAP@50-95": 44.9, "Speed (GPU ms)": 1.20},
    {"Model": "YOLOv8m", "Size": "52MB", "Params": "25.9M", "mAP@50-95": 50.2, "Speed (GPU ms)": 1.83},
    {"Model": "YOLOv8l", "Size": "87MB", "Params": "43.7M", "mAP@50-95": 52.9, "Speed (GPU ms)": 2.39},
    {"Model": "YOLOv8x", "Size": "218MB", "Params": "68.2M", "mAP@50-95": 53.9, "Speed (GPU ms)": 3.53},
])

print("YOLOv8 Model Family")
yolo_df

In [ ]:
try:
    from ultralytics import YOLO
    import cv2

    YOLO_MODEL = "yolov8s.pt"  # Options: yolov8n/s/m/l/x.pt
    yolo_model = YOLO(YOLO_MODEL)
    print(f"YOLOv8 loaded: {YOLO_MODEL}")

    # Run detection on test image
    test_image_yolo = load_image_from_url(IMAGE_URL)
    results = yolo_model(test_image_yolo, verbose=False)

    rows = []
    for box in results[0].boxes:
        class_id = int(box.cls[0])
        confidence = float(box.conf[0])
        bbox = box.xyxy[0].tolist()
        rows.append({
            "Label": yolo_model.names[class_id],
            "Score": f"{confidence:.4f}",
            "Box": f"({bbox[0]:.0f}, {bbox[1]:.0f}, {bbox[2]:.0f}, {bbox[3]:.0f})",
        })

    yolo_det_df = pd.DataFrame(rows)
    print(f"\nYOLO detected {len(rows)} objects:")
    display(yolo_det_df)

except ImportError:
    print("ultralytics not installed. Skipping YOLOv8 section.")
    print("Install with: pip install ultralytics")
    yolo_model = None

In [ ]:
# Visualize YOLO detections (if available)
if yolo_model is not None:
    import matplotlib.pyplot as plt

    annotated_img = results[0].plot()
    plt.figure(figsize=(12, 8))
    plt.imshow(cv2.cvtColor(annotated_img, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("YOLOv8 Object Detection", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("Skipped: YOLOv8 not available.")

## Performance Benchmarking

Let's compare DETR inference speed and, if available, YOLO speed.

In [ ]:
def benchmark_detection(
    image: Image.Image, n_runs: int = 5
) -> pd.DataFrame:
    """Benchmark DETR (and optionally YOLO) inference speed.

    Args:
        image: PIL Image to use for benchmarking.
        n_runs: Number of runs to average over.

    Returns:
        DataFrame with model, mean time, and FPS.
    """
    rows = []

    # Benchmark DETR
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        detector(image, threshold=0.7)
        times.append(time.perf_counter() - start)
    mean_detr = np.mean(times)
    rows.append({
        "Model": "DETR-ResNet-50",
        "Mean (ms)": f"{mean_detr * 1000:.1f}",
        "FPS": f"{1 / mean_detr:.1f}",
    })

    # Benchmark YOLO if available
    if yolo_model is not None:
        times = []
        for _ in range(n_runs):
            start = time.perf_counter()
            yolo_model(image, verbose=False)
            times.append(time.perf_counter() - start)
        mean_yolo = np.mean(times)
        rows.append({
            "Model": "YOLOv8s",
            "Mean (ms)": f"{mean_yolo * 1000:.1f}",
            "FPS": f"{1 / mean_yolo:.1f}",
        })

    return pd.DataFrame(rows)


bench_image = load_image_from_url(IMAGE_URL)
print(f"Detection Benchmark ({device})")
benchmark_detection(bench_image)

### DETR vs YOLO Comparison

In [ ]:
comparison_df = pd.DataFrame([
    {"Aspect": "Model Size", "DETR": "159MB (ResNet-50)", "YOLOv8": "6-218MB (n-x)"},
    {"Aspect": "Speed (GPU)", "DETR": "~50ms", "YOLOv8": "~1-4ms"},
    {"Aspect": "Accuracy (COCO)", "DETR": "42.0 mAP", "YOLOv8": "37-54 mAP"},
    {"Aspect": "Architecture", "DETR": "Transformer", "YOLOv8": "CNN"},
    {"Aspect": "Best For", "DETR": "Research, accuracy", "YOLOv8": "Real-time, production"},
    {"Aspect": "Training Complexity", "DETR": "Complex", "YOLOv8": "Simple"},
    {"Aspect": "Edge Deployment", "DETR": "Difficult", "YOLOv8": "Excellent"},
])

print("DETR vs YOLOv8")
comparison_df

**When to use YOLO**: Real-time applications (video, webcam), production deployments, edge devices.

**When to use DETR**: Research projects, when accuracy matters more than speed, transformer-based pipelines.

## Error Analysis

Let's test detection on challenging inputs to understand model limitations.

In [ ]:
def analyze_failure_modes() -> pd.DataFrame:
    """Test object detection on challenging synthetic images.

    Returns:
        DataFrame with test case, description, and detection count.
    """
    test_cases = []

    # Solid color image (no objects)
    solid = Image.new("RGB", (224, 224), color=(128, 128, 128))
    dets = detector(solid, threshold=0.5)
    test_cases.append({"Test": "Solid gray image", "Expected": "0 objects", "Detected": len(dets), "Note": "False positives?" if len(dets) > 0 else "Correct"})

    # Random noise
    noise = Image.fromarray(np.random.randint(0, 255, (224, 224, 3), dtype=np.uint8))
    dets = detector(noise, threshold=0.5)
    test_cases.append({"Test": "Random noise", "Expected": "0 objects", "Detected": len(dets), "Note": "False positives?" if len(dets) > 0 else "Correct"})

    # Very small image
    tiny = Image.new("RGB", (32, 32), color=(200, 100, 50))
    dets = detector(tiny, threshold=0.5)
    test_cases.append({"Test": "Tiny image (32x32)", "Expected": "0 objects", "Detected": len(dets), "Note": "May hallucinate on upscale"})

    # Very large image (white)
    white = Image.new("RGB", (1920, 1080), color=(255, 255, 255))
    dets = detector(white, threshold=0.5)
    test_cases.append({"Test": "White 1080p image", "Expected": "0 objects", "Detected": len(dets), "Note": "False positives?" if len(dets) > 0 else "Correct"})

    return pd.DataFrame(test_cases)


print("Error Analysis: Challenging Inputs")
analyze_failure_modes()

**Observations**: Object detectors can produce false positives on featureless images and may hallucinate objects in noise. Small images are upscaled internally, which can also cause issues. These edge cases matter for production systems where unexpected inputs are common.

## State-of-the-Art Models

In [ ]:
sota_df = pd.DataFrame([
    {"Model": "DETR", "Size": "159MB", "mAP": 42.0, "Speed": "Medium", "Best For": "Learning transformers"},
    {"Model": "YOLOv8", "Size": "6-218MB", "mAP": 53.9, "Speed": "Very fast", "Best For": "Real-time, production"},
    {"Model": "DINO", "Size": "1.1GB", "mAP": 63.3, "Speed": "Slow", "Best For": "High accuracy, research"},
    {"Model": "Grounding DINO", "Size": "680MB", "mAP": 57.0, "Speed": "Slow", "Best For": "Open-vocabulary detection"},
    {"Model": "SAM", "Size": "375MB-2.4GB", "mAP": "N/A", "Speed": "Very slow", "Best For": "Universal segmentation"},
])

print("State-of-the-Art Object Detection Models (COCO benchmark)")
print("Note: SAM is a segmentation model, not directly comparable on mAP.\n")
sota_df

## Exercises

1. **Custom Images**: Test with your own images. What objects are detected? Try different scenes.

2. **Threshold Tuning**: Find the optimal threshold for different image types (indoor vs outdoor, crowded vs sparse).

3. **Object Counter**: Build a function that counts total objects across multiple images and returns summary statistics.

4. **Model Comparison**: Compare DETR-ResNet-50 vs DETR-ResNet-101 (if you have GPU). Measure accuracy and speed differences.

5. **Visualization**: Enhance the bounding box drawing with different colors per class and a legend.

## Key Takeaways

- **Object detection** finds both what and where objects are in an image
- **DETR** is a modern transformer-based detector that eliminates anchor boxes and NMS
- **Confidence threshold** controls the precision/recall tradeoff
- **Bounding boxes** use (xmin, ymin, xmax, ymax) format in pixel coordinates
- **YOLOv8** is the production standard for real-time detection (100+ FPS on GPU)
- Detection is more computationally intensive than classification

## Next Steps & Resources

**Next Notebook**: Notebook 06 -- OCR for text extraction from images

**Documentation**:
- [DETR Paper](https://arxiv.org/abs/2005.12872)
- [HuggingFace Object Detection Guide](https://huggingface.co/docs/transformers/tasks/object_detection)
- [Ultralytics YOLOv8 Docs](https://docs.ultralytics.com/)
- [COCO Dataset Categories](https://cocodataset.org/#explore)